In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
sc = spark.sparkContext

In [3]:
orders_base_rdd = sc.textFile("data/retail_db_orders_part-00000")
customers_base_rdd = sc.textFile("data/retail_db_customers_part-00000")
order_items_base_rdd = sc.textFile("data/retail_db_order_items_part-00000")

## State that has most number of orders in CLOSED status

In [4]:
closed_order = orders_base_rdd.filter(lambda x: x.split(",")[3] == 'CLOSED')

In [5]:
closed_order.take(2)

['1,2013-07-25 00:00:00.0,11599,CLOSED', '4,2013-07-25 00:00:00.0,8827,CLOSED']

In [6]:
closed_cols = closed_order.map(lambda x: (x.split(",")[2], x.split(",")[3]))

In [7]:
closed_cols.take(2)

[('11599', 'CLOSED'), ('8827', 'CLOSED')]

In [8]:
customers_rdd_col = customers_base_rdd.map(lambda x: (x.split(",")[0], x.split(",")[7]))

In [ ]:
customers_rdd_col.take(5)


[('1', 'TX'), ('2', 'CO'), ('3', 'PR'), ('4', 'CA'), ('5', 'PR')]

In [10]:
joined_rdd =  closed_cols.join(customers_rdd_col)

In [11]:
joined_rdd.take(2)

[('5834', ('CLOSED', 'PR')), ('5834', ('CLOSED', 'PR'))]

In [12]:
map_rdd = joined_rdd.map(lambda x: (x[1][1], 1))

In [13]:
map_rdd.reduceByKey(lambda x,y: x+y).take(1)

[('PR', 2891)]

In [14]:
spark.stop()